# snapshot

> Flat namespace serializer

In [ ]:
#| default_exp snapshot

In [ ]:
#| hide
from nbdev.showdoc import *

In [ ]:
from paar.snapshot import snapshot, expand, grid_page
import pathlib, numpy as np, pandas as pd
def test_sorted_and_fields():
    rows = snapshot({'b':1, 'a':'hi'})
    assert [r.name for r in rows]==['a','b']
    assert rows[0].type=='str' and rows[0].value=="'hi'" and rows[0].path=='a'
    assert rows[1].type=='int' and rows[1].qualifier=='builtins'
def test_skips_hidden_and_dunder():
    rows = snapshot({'x':1, '_ih':2, '__y':3}, hidden=frozenset({'_ih'}))
    assert [r.name for r in rows]==['x']
def test_shape_populated():
    assert snapshot({'L':[1,2,3]})[0].shape=='3'
def test_container_flag_and_accessor():
    rows = snapshot({'d':{'a':1}, 'n':5})
    d, n = rows[0], rows[1]
    assert d.is_container is True and d.accessor==('d',)
    assert n.is_container is False and n.accessor==('n',)
def test_expand_dict_then_list():
    ns = {'d': {'x': 1, 'y': [10,20]}}
    kids = expand(ns, ('d',))
    assert [k.name for k in kids]==["'x'", "'y'"]
    assert kids[1].is_container and kids[1].accessor==('d',1) and kids[1].path=="d['y']"
    gk = expand(ns, ('d',1))
    assert [g.name for g in gk]==['0','1'] and gk[0].value=='10' and gk[0].path=="d['y'][0]"
def test_expand_truncates():
    kids = expand({'L': list(range(250))}, ('L',))
    assert len(kids)==101 and kids[-1].is_error is False and 'more' in kids[-1].value
def test_expand_non_container_empty():
    assert expand({'n':5}, ('n',))==[]
def test_gridables_are_grid_and_container():
    rows = snapshot({'arr': np.arange(4), 'df': pd.DataFrame({'a':[1]}), 'lst':[1,2]})
    by = {r.name:r for r in rows}
    assert by['arr'].is_grid and by['arr'].is_container   # both: grid link + tree
    assert by['df'].is_grid and by['df'].is_container
    assert by['lst'].is_container and not by['lst'].is_grid
def test_expand_ndarray_detail():
    kids = expand({'arr': np.arange(6).reshape(2,3)}, ('arr',))
    names = [k.name for k in kids]
    assert names[:3]==['shape','dtype','size'] and 'mean' in names and names[-2:]==['0','1']
    row0 = expand({'arr': np.arange(6).reshape(2,3)}, ('arr', names.index('0')))  # first row [0 1 2]
    elems = {k.name:k.value for k in row0}
    assert elems['0']=='0' and elems['2']=='2'   # numpy scalars render cleanly
def test_expand_dataframe_columns():
    kids = expand({'df': pd.DataFrame({'x':[1,2],'y':[3,4]})}, ('df',))
    names = [k.name for k in kids]
    assert 'shape' in names and names[-2:]==['x','y']
    col = expand({'df': pd.DataFrame({'x':[1,2],'y':[3,4]})}, ('df', names.index('x')))
    assert col[0].name=='dtype'   # column expands as a Series
def test_grid_page_walk():
    p = grid_page({'arr': np.arange(6).reshape(2,3)}, ('arr',))
    assert p['nrows']==2 and p['ncols']==3 and p['cells'][1]==['3','4','5']
def test_grid_page_non_gridable_none():
    assert grid_page({'n':5}, ('n',)) is None
def test_value_uses_provider():
    by = {r.name:r for r in snapshot({'p': pathlib.Path('/tmp/x'), 'n': 5})}
    assert by['p'].value==str(pathlib.Path('/tmp/x'))
    assert by['n'].value=='5'
for t in [test_sorted_and_fields,test_skips_hidden_and_dunder,test_shape_populated,
          test_container_flag_and_accessor,test_expand_dict_then_list,
          test_expand_truncates,test_expand_non_container_empty,
          test_gridables_are_grid_and_container,test_expand_ndarray_detail,
          test_expand_dataframe_columns,test_grid_page_walk,test_grid_page_non_gridable_none,
          test_value_uses_provider]: t()

In [ ]:
#| export
from paar.core import VarInfo
from paar.reprs import get_shape, get_dtype
from paar.resolvers import resolve_for
from paar.grid import is_gridable, array_page
from paar.providers import value_str

MAX_CHILDREN = 100

def _var_info(name, v, accessor, path):
    "Build a VarInfo for one value at the given accessor/path."
    try:
        return VarInfo(name=name, type=type(v).__name__,
                       qualifier=getattr(type(v), '__module__', '') or '',
                       value=value_str(v), shape=get_shape(v), dtype=get_dtype(v),
                       is_container=resolve_for(v) is not None,   # gridables are also expandable
                       is_grid=is_gridable(v), path=path, accessor=accessor)
    except Exception as e:
        return VarInfo(name=name, type='?', value=f'<error {e}>', is_error=True,
                       path=path, accessor=accessor)

def snapshot(ns, hidden=frozenset()):
    "namespace dict -> sorted list[VarInfo], skipping hidden and dunder names."
    keys = sorted(k for k in ns if k not in hidden and not k.startswith('__'))
    return [_var_info(k, ns[k], (k,), k) for k in keys]

def _walk(ns, accessor):
    "Resolve a positional accessor (name, *idxs) to (object, readable_path). Raises KeyError/IndexError on bad path."
    name, *idxs = accessor
    obj, path = ns[name], name
    for i in idxs:
        r = resolve_for(obj)
        if r is None: raise KeyError(accessor)
        _, step, obj = r.children(obj)[i]
        path += step
    return obj, path

def expand(ns, accessor):
    "Return one level of children of the value at accessor, as VarInfo (capped at MAX_CHILDREN)."
    try:
        obj, path = _walk(ns, tuple(accessor))
    except Exception:
        return []
    r = resolve_for(obj)
    if r is None: return []
    kids = r.children(obj)
    out = [_var_info(nm, child, tuple(accessor)+(i,), path+step)
           for i,(nm, step, child) in enumerate(kids[:MAX_CHILDREN])]
    if len(kids) > MAX_CHILDREN:
        out.append(VarInfo(name='…', type='', value=f'{len(kids)-MAX_CHILDREN} more',
                           path=path, accessor=tuple(accessor)))
    return out

def grid_page(ns, accessor, roff=0, coff=0, rows=50, cols=50):
    "Walk accessor to a gridable object and return a page dict, or None."
    try:
        obj, _ = _walk(ns, tuple(accessor))
        if not is_gridable(obj): return None
        return array_page(obj, roff, coff, rows, cols)
    except Exception:
        return None

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()